In [ ]:
import sys
from pyspark.sql import SparkSession
# Añadir la carpeta 'src' al PATH para poder importar los módulos
# La ruta es relativa al directorio 'work' del contenedor
sys.path.append('/home/jovyan/work/src') 

# Importar la función que necesitas de tu módulo
import Consulta_sql as sql_gen

In [ ]:
# 1. INICIALIZAR SPARK
spark = (SparkSession.builder
    .appName("MainPipeline")
    .getOrCreate()
)

# 2. CARGAR DATOS (Simulación de Extracción)
df_clients = spark.read.csv("/home/jovyan/work/data/CLIENTS.csv", header=True, inferSchema=True)
df_behaviour = spark.read.csv("/home/jovyan/work/data/BEHAVIOURAL.csv", header=True, inferSchema=True)

In [ ]:
# 3. LLAMAR A FUNCIÓN DE MÓDULO (Limpieza/Transformación)
# Llamamos a la función que crea las vistas SQL:
sql_gen.create_temp_views(spark, df_clients, df_behaviour)


In [ ]:
columnas_deseadas = [
    "CLIENT_ID",
    "AGE_IN_YEARS",
    "CREDIT_CARD_PAYMENT"  # Solo esta del DataFrame 'behaviour'
]

sql_query = sql_gen.get_dynamic_segmentation_query(columnas_deseadas)
df_final = spark.sql(sql_query)

print("\n--- Consulta SQL Generada ---")
print(sql_query)
print("\n--- Resultados de la Consulta Dinámica ---")
df_final.show()

spark.stop()